# Hiver AI Customer Support Agent — Kaggle Dual T4 GPU Pipeline
### End-to-End QLoRA Fine-Tuning, Multi-Signal Escalation & Evaluation Benchmark

**Author**: Hiver SDE Intern Candidate (12 LPA | 2027 Batch)  
**Hardware Target**: Kaggle Dual Tesla T4 (16GB x 2 VRAM) with 4-Bit QLoRA  
**Dataset**: 5,000 Verified Conversational Resolution Pairs (`amazon_conversations_formatted.jsonl`)

In [ ]:
# 1. Install & Verify Essential Dependencies
!pip install -q -U "transformers>=4.44.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "trl>=0.9.6" "accelerate>=0.33.0" "datasets>=2.20.0" "rouge-score>=0.1.2"
import torch
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU [{i}]        : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB VRAM)')
else:
    print('WARNING: GPU Accelerator not detected. Please verify Accelerator is set to GPU in Kaggle settings.')


In [ ]:
# 2. Clone Repository or Locate Mounted Dataset
import os, sys
!git clone https://github.com/your-username/Hiver-Assignment.git /kaggle/working/Hiver-Assignment || true
sys.path.insert(0, '/kaggle/working/Hiver-Assignment')

TRAIN_DATA = '/kaggle/working/Hiver-Assignment/data/processed/amazon_conversations_formatted.jsonl'
EVAL_DATA = '/kaggle/working/Hiver-Assignment/data/golden_set/golden_eval_200.jsonl'

# Check fallback local paths if running directly inside repo
if not os.path.exists(TRAIN_DATA):
    TRAIN_DATA = 'data/processed/amazon_conversations_formatted.jsonl'
    EVAL_DATA = 'data/golden_set/golden_eval_200.jsonl'

print('Training data path  :', TRAIN_DATA, '| Exists:', os.path.exists(TRAIN_DATA))
print('Evaluation data path:', EVAL_DATA, '| Exists:', os.path.exists(EVAL_DATA))


In [ ]:
# 3. Run Fast 5-Step Dry-Run (Guarantees zero wasted GPU hours)
!python /kaggle/working/Hiver-Assignment/scripts/train_support_slm.py \
    --train_data {TRAIN_DATA} \
    --dry_run


In [ ]:
# 4. Full Fine-Tuning Execution with 4-Bit QLoRA
!python /kaggle/working/Hiver-Assignment/scripts/train_support_slm.py \
    --model_name "meta-llama/Llama-3.2-3B-Instruct" \
    --train_data {TRAIN_DATA} \
    --eval_data {EVAL_DATA} \
    --output_dir "/kaggle/working/amazon_support_slm" \
    --batch_size 2 \
    --grad_accum_steps 4 \
    --learning_rate 2e-4 \
    --epochs 1


In [ ]:
# 5. Execute 200-Sample Golden Benchmark Evaluation
from scripts.run_evaluation import run_benchmark
from scripts.evaluate_judge_agreement import main as run_judge_agreement

print('=== RUNNING 3-MODEL BENCHMARK ===')
run_benchmark()

print('\n=== RUNNING HUMAN VS JUDGE CALIBRATION ===')
run_judge_agreement()
